# Flipster on a Colab GPU

Builds Flipster with the **CUDA** engine on Colab's NVIDIA GPU, runs the C++ and Python test suites (including the
CPU ↔ CUDA parity tests), benchmarks every backend, evaluates accuracy on Middlebury, renders the sample flipbook, and
optionally serves the web app.

**Before you start:** *Runtime → Change runtime type → **T4 GPU*** (any NVIDIA GPU works). Then *Runtime → Run all*.
Cell 2 either clones the repo from GitHub or asks you to upload `Flipster-v2.zip`. The whole run takes about
10–15 minutes on a T4; the last cell downloads a zip with every result.

In [ ]:
# 1. Check the GPU and define a helper that streams a command's output and stops on failure.
import os
import shutil
import subprocess
import sys
import time


def run(cmd, cwd=None, env=None):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        env={**os.environ, **(env or {})},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {cmd}")


if shutil.which("nvidia-smi") is None:
    raise SystemExit("No NVIDIA GPU. Use Runtime > Change runtime type > T4 GPU, then run again.")
run("nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv")
run("nvcc --version | tail -2")
GPU = (
    subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True)
    .stdout.strip()
    .splitlines()[0]
)
GPU_SLUG = "colab-" + GPU.lower().replace("nvidia", "").strip().replace(" ", "-")
print("GPU:", GPU)

In [ ]:
# 2. Get the code: clone from GitHub, or fall back to uploading Flipster-v2.zip.
REPO = "/content/Flipster"
GITHUB = "https://github.com/lolaitan/Flipster.git"  # works once the branch below is pushed and the repo is public
BRANCH = "rewrite"

if not os.path.exists(f"{REPO}/pyproject.toml"):
    r = subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "-b", BRANCH, GITHUB, REPO],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    if r.returncode != 0:
        from google.colab import files

        print("Clone didn't work (private repo, or the branch isn't pushed yet). Upload Flipster-v2.zip instead:")
        uploaded = files.upload()
        name = next(iter(uploaded))
        run(f"unzip -q -o '{name}' -d /content && rm '{name}'")
os.chdir(REPO)
run("ls")

In [ ]:
# 3. Build and install the Python package with the CUDA engine (about 3-5 minutes).
run("pip install -q -U cmake ninja")  # CMake >= 3.24 for CUDA_ARCHITECTURES=native
run(
    'pip install ".[server,dev]" --config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES=native',
    env={"FLIPSTER_ENABLE_CUDA": "ON", "CUDACXX": "/usr/local/cuda/bin/nvcc"},
)
run('python -c "import flipster, json; print(json.dumps(flipster.device_info(), indent=2))"', cwd="/content")

In [ ]:
# 4. C++ tests, including CPU <-> CUDA parity for the optimized and naive kernels.
run(
    "cmake -S . -B build/colab -G Ninja -DFLIPSTER_ENABLE_CUDA=ON -DFLIPSTER_BUILD_TESTS=ON "
    "-DCMAKE_CUDA_ARCHITECTURES=native -DCMAKE_BUILD_TYPE=Release"
)
run("cmake --build build/colab")
run("ctest --test-dir build/colab --output-on-failure")
for backend, variant in [("cuda", "optimized"), ("cuda", "naive"), ("cpu", "optimized")]:
    run(f"build/colab/flipster_bench --backend {backend} --variant {variant} --size 1920x1080 --reps 10")

In [ ]:
# 5. Python tests: NumPy reference, C++ and CUDA engines, parity, API.
run("python -m pytest -q -p no:cacheprovider")

In [ ]:
# 6. Benchmark every backend (480p to 4K) and save the table for the README.
from IPython.display import Markdown, display

out = f"bench/results/{GPU_SLUG}.md"
run(f"python bench/run_bench.py --sizes 480p,720p,1080p,4k --reps 10 --out {out}")
display(Markdown(open(out).read()))

In [ ]:
# 7. Accuracy on Middlebury. Optional: if vision.middlebury.edu is unreachable, this cell
#    reports it and the rest of the notebook keeps going.
#    Manual fallback: download other-color-twoframes.zip, other-gt-flow.zip and other-gt-interp.zip from
#    https://vision.middlebury.edu/flow/data/ in a browser, set UPLOAD_MIDDLEBURY = True and run this cell again.
from IPython.display import Markdown, display

UPLOAD_MIDDLEBURY = False

if UPLOAD_MIDDLEBURY:
    from google.colab import files

    for name in files.upload():
        run(f"unzip -q -o '{name}' -d eval/data && rm '{name}'")
    cmd = "python eval/middlebury.py --backend cuda --out eval/results.md"
else:
    cmd = "python eval/middlebury.py --download --backend cuda --out eval/results.md"
try:
    run(cmd)
    display(Markdown(open("eval/results.md").read()))
except RuntimeError:
    print("\nMiddlebury evaluation skipped (see the message above). Everything else still runs.")

In [ ]:
# 8. Render the sample flipbook on the GPU vs the CPU, and regenerate the README figures.
import glob

from IPython.display import Image

from flipster import RenderOptions, render
from flipster.preprocess import load_rgb

pages = [load_rgb(p) for p in sorted(glob.glob("samples/scans/*.jpg"))]
for backend in ["cuda", "cpu"]:
    s = render(pages, RenderOptions(method="flow", inbetweens=3, backend=backend)).summary()
    print(
        f"{backend:>4}: {s['frames']} frames in {s['total_ms'] / 1000:.1f} s "
        f"(flow {s['flow_ms'] / 1000:.2f} s, synthesis {s['synth_ms'] / 1000:.2f} s, "
        f"preprocessing {s['preprocess_ms'] / 1000:.2f} s)"
    )
run("python docs/make_figures.py --backend cuda")
Image(open("docs/demo.gif", "rb").read())

### Optional: use the web app from Colab

The next cell builds the React app, starts the server on the GPU, and prints a link that opens it through Colab's proxy.
The render progress bar may only update at the end (Colab's proxy buffers streamed responses).

In [ ]:
# 9. (optional) Serve the web app.
import re
import urllib.request

from google.colab.output import eval_js


def node_ok():
    try:
        v = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return False
    m = re.match(r"v(\d+)\.(\d+)", v)
    major, minor = (int(m[1]), int(m[2])) if m else (0, 0)
    return major > 22 or (major == 22 and minor >= 12) or (major == 20 and minor >= 19)


if not node_ok():  # Vite 8 needs Node 20.19+ / 22.12+
    run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs")
run("npm ci --no-audit --no-fund && npm run build", cwd="web")

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--app-dir", "server", "--port", "8000"],
    cwd=REPO,
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
print("Open Flipster:", eval_js("google.colab.kernel.proxyPort(8000)"))

In [ ]:
# 10. (optional) Profile the LK kernels with Nsight Compute, if this runtime allows GPU counters.
if shutil.which("ncu"):
    try:
        run(
            "ncu --kernel-name regex:k_ --launch-skip 40 --launch-count 12 --section SpeedOfLight "
            "build/colab/flipster_bench --backend cuda --size 1920x1080 --reps 1"
        )
    except RuntimeError as e:
        print("Nsight Compute could not collect counters on this runtime:", e)
else:
    print("ncu is not installed on this runtime; skipping.")

In [ ]:
# 11. Download everything worth keeping (benchmark, accuracy, figures).
from google.colab import files

extras = "eval/results.md" if os.path.exists("eval/results.md") else ""
run(f"zip -q -r /content/flipster-{GPU_SLUG}-results.zip bench/results docs/*.png docs/*.gif {extras}")
files.download(f"/content/flipster-{GPU_SLUG}-results.zip")